In [ ]:
import json
import csv
import os
import time
import requests
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


def scrape_ids(sale_id):
    api_url = "https://g3-api.g3r.co.uk/lot"
    params = {
        "orderby": "lot_num",
        "orderbydir": "asc",
        "pagenum": 1,
        "perpage": 5000,
        "sale_ids": sale_id,
        "tracked_only": "false"
    }
    response = requests.get(api_url, params=params)
    lot_ids = []

    if response.status_code == 200:
        data = response.json()
        lot_ids = [lot["lot_id"] for lot in data.get("data", [])]

        os.makedirs("database", exist_ok=True)
        ids_file = f"database/auctions_id.json"
        with open(ids_file, "w", encoding="utf-8") as f:
            json.dump({"count": len(lot_ids), "ids": lot_ids}, f, indent=4)
        print(f"✅ Saved {len(lot_ids)} IDs to '{ids_file}'")
    else:
        print(f"❌ Failed to fetch IDs — status {response.status_code}")

    return lot_ids


def fetch_vehicle_details(lot_ids, sale_id):
    output_file = f"database/vehiclesData.json"
    all_vehicles = []

    print(f"📦 Fetching {len(lot_ids)} vehicle details...")

    for vid in tqdm(lot_ids, desc="Fetching vehicles"):
        try:
            response = requests.get(f"https://g3-api.g3r.co.uk/lot/{vid}")
            if response.status_code == 200:
                all_vehicles.append(response.json())
            else:
                print(f"⚠️ Failed for lot {vid} — status {response.status_code}")
        except Exception as e:
            print(f"⚠️ Error fetching lot {vid}: {e}")
        time.sleep(0.1)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_vehicles, f, indent=4)
    print(f"✅ Saved {len(all_vehicles)} vehicles to '{output_file}'")


def scrape_images():
    # Load auction IDs
    with open('database/auctions_id.json', 'r', encoding='utf-8') as f:
        auction_ids = json.load(f)['ids']

    csv_file = 'g3_images.csv'
    fieldnames = ['id', 'Images', 'Damaged_images']

    options = Options()
    # options.add_argument("--headless")  # Uncomment for headless mode
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    rows = []

    for auction_id in auction_ids:
        url = f"https://www.g3remarketing.co.uk/lots/{auction_id}"
        driver.get(url)
        time.sleep(2)

        try:
            tabs = WebDriverWait(driver, 5).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ul.nav-tabs li a"))
            )
        except:
            print(f"⚠️ Tabs not found for ID {auction_id}")
            continue

        marketing_urls = []
        damage_urls = []

        for tab in tabs:
            tab_text = tab.text.strip()
            tab.click()
            time.sleep(1)  # wait for images to load

            if 'Marketing' in tab_text:
                images = driver.find_elements(By.CSS_SELECTOR, "div.image-gallery-slide img")
                marketing_urls = [img.get_attribute('src') for img in images if img.get_attribute('src')]

            elif 'Condition' in tab_text:
                images = driver.find_elements(By.CSS_SELECTOR, "img.image-gallery-image")
                damage_urls = [img.get_attribute('src') for img in images if img.get_attribute('src')]

        rows.append({
            'id': auction_id,
            'Images': ','.join(marketing_urls),
            'Damaged_images': ','.join(damage_urls)
        })

        print(f"✅ Extracted images for ID {auction_id}")


    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    driver.quit()
    print(f"✅ All images saved to '{csv_file}'")


def main():
    sale_id = 1394  
    lot_ids = scrape_ids(sale_id)
    if lot_ids:
        fetch_vehicle_details(lot_ids, sale_id)
        scrape_images() 

if __name__ == "__main__":
    main()


✅ Saved 72 IDs to 'database/auctions_id.json'
📦 Fetching 72 vehicle details...


Fetching vehicles: 100%|██████████| 72/72 [01:07<00:00,  1.07it/s]


✅ Saved 72 vehicles to 'database/vehiclesData.json'
✅ Extracted images for ID 121638
✅ Extracted images for ID 119153
✅ Extracted images for ID 120017
✅ Extracted images for ID 120960
✅ Extracted images for ID 120957
✅ Extracted images for ID 121625
✅ Extracted images for ID 121620
✅ Extracted images for ID 120012
✅ Extracted images for ID 119924
✅ Extracted images for ID 119928
✅ Extracted images for ID 119927
✅ Extracted images for ID 121046
✅ Extracted images for ID 120920
✅ Extracted images for ID 119925
✅ Extracted images for ID 120923
✅ Extracted images for ID 120918
✅ Extracted images for ID 120929
✅ Extracted images for ID 120927
✅ Extracted images for ID 120830
✅ Extracted images for ID 120829
✅ Extracted images for ID 121682
✅ Extracted images for ID 121683
✅ Extracted images for ID 120833
✅ Extracted images for ID 121294
✅ Extracted images for ID 121704
✅ Extracted images for ID 119282
✅ Extracted images for ID 121157
✅ Extracted images for ID 120355
✅ Extracted images for I

In [ ]:
import json
import csv

with open('database/vehiclesData.json', 'r', encoding='utf-8') as f:
    vehicles = json.load(f)

csv_file = 'g3remarketing.csv'

rows = []
for vehicle in vehicles:
    id = vehicle.get('lot_id', "")
    reg = vehicle.get('vehicle', {}).get('vrm')
    start_dt = vehicle.get('sale', {}).get('start_date_string')  
    start_date = start_time = ''
    if start_dt:
        parts = start_dt.split(' ')
        if len(parts) == 2:
            start_date, start_time = parts
    mileage = vehicle.get('vehicle', {}).get('mileage', '')
    make = vehicle.get('vehicle', {}).get('make_name', '')
    Model = vehicle.get('vehicle', {}).get('model_name', '')
    Variant = vehicle.get('vehicle', {}).get('deriv_name', '')
    manufactured = vehicle.get('vehicle', {}).get('manufactured', '')
    num_keys = vehicle.get('vehicle', {}).get('num_keys', '')
    previous_keepers = vehicle.get('vehicle', {}).get('previous_keepers', '')
    engine_cc = vehicle.get('vehicle', {}).get('engine_cc', '')
    engine_litre  = round(engine_cc/1000,1) if engine_cc else ''
    mot_expiry = vehicle.get('vehicle', {}).get('mot_expiry_string', '')
    first_registered_string = vehicle.get('vehicle', {}).get('first_registered_string', '')
    doors = vehicle.get('vehicle', {}).get('number_of_doors', '')
    seats = vehicle.get('vehicle', {}).get('seats', '')
    mileage_war = vehicle.get('vehicle', {}).get('mileage_warranted', '')
    mileage_warranted = "Warranted" if mileage == True else 'Not'
    location_raw = vehicle.get('vehicle', {}).get('location', '').strip()
    if location_raw.upper().startswith("G3 "):
       location = location_raw[3:].strip()
    else:
        location = location_raw
    euro_status = vehicle.get('vehicle', {}).get('euro_status', {}).get('title', '')
    fuel_type = vehicle.get('vehicle', {}).get('fuel_type', {}).get('title', '')
    transmission_type= vehicle.get('vehicle', {}).get('transmission_type', {}).get('title', '')
    colour = vehicle.get('vehicle', {}).get('colour', {}).get('title', '')
    bodystyle = vehicle.get('vehicle', {}).get('bodystyle', {}).get('title', '')
    v5_status = vehicle.get('vehicle', {}).get('v5_status', {}).get('title', '')
    vat_status = vehicle.get('vehicle', {}).get('vat_status', {}).get('title', '')
    vehicle_grade = vehicle.get('vehicle', {}).get('vehicle_grade', {}).get('title', '')
    sale_name = vehicle.get('sale', {}).get('sale_name', '')
    runner_status = vehicle.get('vehicle', {}).get('runner_status', {}).get('title', '')
    vehicle_type = vehicle.get('vehicle', {}).get('vehicle_type', {}).get('title', '')
    chassis = vehicle.get('vehicle', {}).get('chassis', '')
    nonRunner = "No" if runner_status == "Runner" else "Yes"
    service_mileages = vehicle.get('vehicle', {}).get('service_mileages', [])
    service_dates = ",".join([s.get("date", "")[:10] for s in service_mileages])
    service_count = len(service_mileages)
    last_service = None
    if service_mileages:
        last_service = max(service_mileages, key=lambda x: x.get("date", ""))
    last_service_date = last_service.get("date", "")[:10] if last_service else ""
    last_service_mileage = last_service.get("mileage", "") if last_service else ""
    avid_spec = vehicle.get('vehicle', {}).get('avid_spec', '')
    
    if reg:  
        rows.append({'Reg': reg,
                     'Lot': id, 
                     'Auction Name': sale_name, 
                     'Title':f'{make} {Model} {Variant}', 
                     'Make': make, 
                     'Model': Model, 
                     'Variant': Variant, 
                     'Year': manufactured, 
                     'Start date': start_date, 
                     'Start time': start_time, 
                     'Mileage': mileage,
                     'Former Keepers': previous_keepers,
                     'CC': engine_litre,
                     'MOT Expiry Date': mot_expiry ,
                     'D.O.R': first_registered_string,
                     'Doors':doors,
                     'Seats':seats,
                     'Mileage Warranted':mileage_warranted,
                     'Center':location,
                     'Euro Status':euro_status,
                     'Fuel Type':fuel_type,
                     'Transmission':transmission_type,
                     'Colour':colour,
                     'Body Type':bodystyle,
                     'V5':v5_status,
                     'VAT Status':vat_status,
                     'Grade':vehicle_grade,
                     'Non Runner':nonRunner,
                     'Service History':service_dates,
                     'No of services':service_count,
                     'Last Service':last_service_date,
                     'Last service mileage':last_service_mileage,
                     'VIN':chassis,
                     'Vehicle Type':vehicle_type,
                     'Equipment':avid_spec,
                     'Keys':num_keys,
                     
                     
                     
                      
                     'id':id, 
                     
                     })


if rows:
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Reg', 
        'Lot',
        'Auction Name',
        'Title',
        'Make',
        'Model',
        'Variant',
        'Year',
        'Start date',
        'Start time', 
        'Mileage',
        'Former Keepers',
        'CC',
        'MOT Expiry Date',
        'D.O.R',
        'Doors',
        'Seats',
        'Mileage Warranted',
        'Center',
        'Euro Status',
        'Fuel Type',
        'Transmission',
        'Colour',
        'Body Type',
        'V5',
        'VAT Status',
        'Grade',
        'Non Runner',
        'Service History',
        'No of services',
        'Last Service',
        'Last service mileage',
        'VIN',
        'Vehicle Type',
        'Equipment',
        'Keys',
        
        
        
        
        'id'
        ])
        writer.writeheader()
        writer.writerows(rows)

print(f"✅ Extracted VRM, Start date, Start time and Mileage saved to CSV at '{csv_file}'")





✅ Extracted VRM, Start date, Start time and Mileage saved to CSV at 'g3remarketing.csv'


In [ ]:

import pandas as pd
img_df = pd.read_csv("g3_images.csv")          
data_df = pd.read_csv("g3remarketing.csv")      
merged = pd.merge(data_df, img_df, on="id", how="left")
merged = merged.drop(columns=["id"])
merged.to_csv("g3remarketing_data.csv", index=False, encoding="utf-8")
print("✅ Merge complete — saved as g3remarketing_data.csv")


✅ Merge complete — saved as g3remarketing_data.csv


In [ ]:
import os
import requests
import pandas as pd
from urllib.parse import urlparse, urljoin
from PIL import Image, ImageDraw, ImageFont

df = pd.read_csv("g3remarketing_data.csv")

reg_img = df[["Reg", "Images"]]
cond_img = df[["Reg", "Damaged_images"]]


def add_watermark_to_image(
        image_path, 
        text="Sourced from G3 Re Marketing",
        font_size=26,         
        padding=25,         
        box_opacity=180       
    ):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

  
        try:
            font = ImageFont.truetype("arial.ttf", font_size)
        except:
            font = ImageFont.load_default()


        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]


        x = image.width - text_width - padding
        y = image.height - text_height - padding


        box_x1 = x - padding
        box_y1 = y - padding
        box_x2 = x + text_width + padding
        box_y2 = y + text_height + padding


        draw.rectangle(
            [box_x1, box_y1, box_x2, box_y2],
            fill=(0, 0, 0, box_opacity)
        )


        draw.text((x, y), text, font=font, fill=(255, 255, 255, 230))


        watermarked = Image.alpha_composite(image, txt_layer).convert("RGB")
        watermarked.save(image_path)

        print(f"Watermarked: {image_path}")

    except Exception as e:
        print(f"Failed watermarking {image_path}: {e}")





def download_images(data, main_folder="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for index, row in data.iterrows():
        reg_no = row["Reg"]

        if pd.isna(row["Images"]) or not str(row["Images"]).strip():
            print(f"No Images for {reg_no}")
            continue

        image_urls = row["Images"].split(",")
        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, url in enumerate(image_urls, start=1):
            url = url.strip()
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            parsed = urlparse(url)
            if not parsed.scheme or not parsed.netloc:
                print(f"Invalid URL: {url}")
                continue

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                img_path = os.path.join(reg_folder, f"{reg_no}_{idx}.jpg")

                with open(img_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(img_path)
                print(f"Downloaded: {img_path}")

            except Exception as e:
                print(f"Failed {url}: {e}")


# -------------------- DOWNLOAD DAMAGED IMAGES --------------------

def download_cond(data, main_folder="Damaged_images"):
    os.makedirs(main_folder, exist_ok=True)

    for index, row in data.iterrows():
        reg_no = row["Reg"]

        if pd.isna(row["Damaged_images"]) or not str(row["Damaged_images"]).strip():
            print(f"No Damage Images for {reg_no}")
            continue

        image_urls = row["Damaged_images"].split(",")
        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)

        for idx, url in enumerate(image_urls, start=1):
            url = url.strip()
            if not url.startswith(("http://", "https://")):
                url = urljoin("https://", url)

            parsed = urlparse(url)
            if not parsed.scheme or not parsed.netloc:
                print(f"Invalid URL: {url}")
                continue

            try:
                response = requests.get(url, stream=True)
                response.raise_for_status()

                img_path = os.path.join(reg_folder, f"{reg_no}_{idx}.jpg")

                with open(img_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(img_path)
                print(f"Downloaded: {img_path}")

            except Exception as e:
                print(f"Failed {url}: {e}")




if __name__ == "__main__":
    download_images(reg_img)
    download_cond(cond_img)
